This notebook generates the **signal features** used for the modeling stage from cleaned OHLCV data.

It computes return-based, volatility, trend, and candle-structure indicators, and prepares a feature matrix suitable for downstream training and backtesting.

In [1]:
# imports
import numpy as np
import os
import sys

sys.path.append(os.path.abspath(".."))
from src.data.data_loader import DataLoad



In [2]:
# import a dataset to test with
data_loader = DataLoad()
df = data_loader.load_single_data("clean", "AAPL")
df.head()

AAPL.csv loaded succesfully


,Close,High,Low,Open,Volume
Date,,,,,
2010-01-04,6.406479,6.421148,6.357685,6.389117,493729600
2010-01-05,6.417557,6.453779,6.383729,6.424143,601904800
2010-01-06,6.315479,6.443004,6.308893,6.417559,552160000
2010-01-07,6.303803,6.346312,6.258002,6.338828,477131200
2010-01-08,6.345711,6.346310,6.258301,6.295420,447610800


In [3]:
# calculate returns and log returns and indicators related to log returns
df["Returns"] = df["Close"].pct_change()
# regular returns will be removed for training, they are there as visual aid
df["Log Returns"] = np.log(df["Close"]).diff()
df["Monthly Log Returns"] = df["Log Returns"].rolling(21).sum()
df["Quarterly Log Returns"] = df["Log Returns"].rolling(63).sum()
df["Semi-annual Log Returns"] = df["Log Returns"].rolling(126).sum()

df



,Close,High,Low,Open,Volume,Returns,Log Returns,Monthly Log Returns,Quarterly Log Returns,Semi-annual Log Returns
Date,,,,,,,,,,
2010-01-04,6.406479,6.421148,6.357685,6.389117,493729600,NaN,NaN,NaN,NaN,NaN
2010-01-05,6.417557,6.453779,6.383729,6.424143,601904800,0.001729,0.001728,NaN,NaN,NaN
2010-01-06,6.315479,6.443004,6.308893,6.417559,552160000,-0.015906,-0.016034,NaN,NaN,NaN
2010-01-07,6.303803,6.346312,6.258002,6.338828,477131200,-0.001849,-0.001850,NaN,NaN,NaN
2010-01-08,6.345711,6.346310,6.258301,6.295420,447610800,0.006648,0.006626,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
2026-01-26,254.936371,256.084232,249.336773,251.013651,55969200,0.029713,0.029280,-0.064255,-0.015226,0.179558
2026-01-27,257.791046,261.464245,257.731160,258.689402,49648300,0.011198,0.011135,-0.058429,-0.016495,0.189899
2026-01-28,255.964447,258.379942,254.038018,257.172195,41288000,-0.007086,-0.007111,-0.064041,-0.046141,0.195861


In [4]:
# Daily volatility and rolling standard deviaton
df["Daily Volatility"] = np.sqrt( (1 / (4*np.log(2))) * ( (np.log( df["High"]/ df["Low"])) **2 ))
# use parkinson estimator for daily volatility
df["Monthly Volatility"] = df["Log Returns"].rolling(21).std()
df["Quarterly Volatility"] = df["Log Returns"].rolling(63).std()
df["Semi-annual Volatility"] = df["Log Returns"].rolling(126).std()
# uses n - 1 in denominator for sample variance
df["Volatility Ratio"] = df["Monthly Volatility"] / df["Quarterly Volatility"]

df





,Close,High,Low,Open,Volume,Returns,Log Returns,Monthly Log Returns,Quarterly Log Returns,Semi-annual Log Returns,Daily Volatility,Monthly Volatility,Quarterly Volatility,Semi-annual Volatility,Volatility Ratio
Date,,,,,,,,,,,,,,,
2010-01-04,6.406479,6.421148,6.357685,6.389117,493729600,NaN,NaN,NaN,NaN,NaN,0.005965,NaN,NaN,NaN,NaN
2010-01-05,6.417557,6.453779,6.383729,6.424143,601904800,0.001729,0.001728,NaN,NaN,NaN,0.006554,NaN,NaN,NaN,NaN
2010-01-06,6.315479,6.443004,6.308893,6.417559,552160000,-0.015906,-0.016034,NaN,NaN,NaN,0.012633,NaN,NaN,NaN,NaN
2010-01-07,6.303803,6.346312,6.258002,6.338828,477131200,-0.001849,-0.001850,NaN,NaN,NaN,0.008416,NaN,NaN,NaN,NaN
2010-01-08,6.345711,6.346310,6.258301,6.295420,447610800,0.006648,0.006626,NaN,NaN,NaN,0.008387,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-26,254.936371,256.084232,249.336773,251.013651,55969200,0.029713,0.029280,-0.064255,-0.015226,0.179558,0.016036,0.011845,0.010290,0.014020,1.151159
2026-01-27,257.791046,261.464245,257.731160,258.689402,49648300,0.011198,0.011135,-0.058429,-0.016495,0.189899,0.008636,0.012116,0.010266,0.014047,1.180237
2026-01-28,255.964447,258.379942,254.038018,257.172195,41288000,-0.007086,-0.007111,-0.064041,-0.046141,0.195861,0.010178,0.012148,0.009876,0.014007,1.230072


In [5]:
# movering averages
df["MA20"] = df["Close"].rolling(20).mean()
df["MA60"] = df["Close"].rolling(60).mean()
df["MA20/MA60"] = df["MA20"] / df["MA60"]
df["Distance from MA20"] = ((df["Close"] - df["MA20"]) / df["MA20"] ) * 100
df["Distance from MA60"] = ((df["Close"] - df["MA60"]) / df["MA60"] ) * 100


df

,Close,High,Low,Open,Volume,Returns,Log Returns,Monthly Log Returns,Quarterly Log Returns,Semi-annual Log Returns,Daily Volatility,Monthly Volatility,Quarterly Volatility,Semi-annual Volatility,Volatility Ratio,MA20,MA60,MA20/MA60,Distance from MA20,Distance from MA60
Date,,,,,,,,,,,,,,,,,,,,
2010-01-04,6.406479,6.421148,6.357685,6.389117,493729600,NaN,NaN,NaN,NaN,NaN,0.005965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-05,6.417557,6.453779,6.383729,6.424143,601904800,0.001729,0.001728,NaN,NaN,NaN,0.006554,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-06,6.315479,6.443004,6.308893,6.417559,552160000,-0.015906,-0.016034,NaN,NaN,NaN,0.012633,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-07,6.303803,6.346312,6.258002,6.338828,477131200,-0.001849,-0.001850,NaN,NaN,NaN,0.008416,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-08,6.345711,6.346310,6.258301,6.295420,447610800,0.006648,0.006626,NaN,NaN,NaN,0.008387,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-26,254.936371,256.084232,249.336773,251.013651,55969200,0.029713,0.029280,-0.064255,-0.015226,0.179558,0.016036,0.011845,0.010290,0.014020,1.151159,260.147674,269.002039,0.967084,-2.003210,-5.228833
2026-01-27,257.791046,261.464245,257.731160,258.689402,49648300,0.011198,0.011135,-0.058429,-0.016495,0.189899,0.008636,0.012116,0.010266,0.014047,1.180237,259.392576,268.816238,0.964944,-0.617415,-4.101386
2026-01-28,255.964447,258.379942,254.038018,257.172195,41288000,-0.007086,-0.007111,-0.064041,-0.046141,0.195861,0.010178,0.012148,0.009876,0.014007,1.230072,258.528181,268.571739,0.962604,-0.991665,-4.694199


In [6]:
# volume averages and ratios
df["VMA20"] = df["Volume"].rolling(20).mean()
df["VMA60"] = df["Volume"].rolling(60).mean()
df["Relative Volume"] = df["Volume"] / df["VMA20"].shift(1)
# use 20 days preceding today to avoid self contamination in denominator
df["Volume Flow Ratio"] = df["VMA20"] / df["VMA60"]
df

,Close,High,Low,Open,Volume,Returns,Log Returns,Monthly Log Returns,Quarterly Log Returns,Semi-annual Log Returns,...,Volatility Ratio,MA20,MA60,MA20/MA60,Distance from MA20,Distance from MA60,VMA20,VMA60,Relative Volume,Volume Flow Ratio
Date,,,,,,,,,,,,,,,,,,,,,
2010-01-04,6.406479,6.421148,6.357685,6.389117,493729600,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-05,6.417557,6.453779,6.383729,6.424143,601904800,0.001729,0.001728,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-06,6.315479,6.443004,6.308893,6.417559,552160000,-0.015906,-0.016034,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-07,6.303803,6.346312,6.258002,6.338828,477131200,-0.001849,-0.001850,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-08,6.345711,6.346310,6.258301,6.295420,447610800,0.006648,0.006626,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-26,254.936371,256.084232,249.336773,251.013651,55969200,0.029713,0.029280,-0.064255,-0.015226,0.179558,...,1.151159,260.147674,269.002039,0.967084,-2.003210,-5.228833,44202740.0,4.664386e+07,1.323155,0.947665
2026-01-27,257.791046,261.464245,257.731160,258.689402,49648300,0.011198,0.011135,-0.058429,-0.016495,0.189899,...,1.180237,259.392576,268.816238,0.964944,-0.617415,-4.101386,45609065.0,4.661989e+07,1.123195,0.978318
2026-01-28,255.964447,258.379942,254.038018,257.172195,41288000,-0.007086,-0.007111,-0.064041,-0.046141,0.195861,...,1.230072,258.528181,268.571739,0.962604,-0.991665,-4.694199,46487705.0,4.614325e+07,0.905259,1.007465


In [7]:
# microstructure
df["Daily Range"] = df["High"] - df["Low"]
# wont be used as indicator, just there to help next calculations
df["Monthly Average Intraday Range"] = df["Daily Range"].rolling(21).mean()
df["True Range"] = np.maximum.reduce([df["Daily Range"],(df["High"] - df["Close"].shift(1)).abs(),(df["Low"] - df["Close"].shift(1)).abs()])
# wont be used as indicator, just there to help next calculations
df["Monthly Average True Range"] = df["True Range"].rolling(21).mean()
df["Daily Candle Body"] = (df["Close"] - df["Open"])
# wont be used as indicator, just there to help next calculations
df["Monthly Average Candle Body"] = df["Daily Candle Body"].abs().rolling(21).mean()
df["Percentage Intraday Range"] = ((df["High"] - df["Low"]) / df["Open"]) * 100
df["Monthly Average Percentage Intraday Range"] = df["Percentage Intraday Range"].rolling(21).mean()


df



,Close,High,Low,Open,Volume,Returns,Log Returns,Monthly Log Returns,Quarterly Log Returns,Semi-annual Log Returns,...,Relative Volume,Volume Flow Ratio,Daily Range,Monthly Average Intraday Range,True Range,Monthly Average True Range,Daily Candle Body,Monthly Average Candle Body,Percentage Intraday Range,Monthly Average Percentage Intraday Range
Date,,,,,,,,,,,,,,,,,,,,,
2010-01-04,6.406479,6.421148,6.357685,6.389117,493729600,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.063463,NaN,NaN,NaN,0.017363,NaN,0.993298,NaN
2010-01-05,6.417557,6.453779,6.383729,6.424143,601904800,0.001729,0.001728,NaN,NaN,NaN,...,NaN,NaN,0.070049,NaN,0.070049,NaN,-0.006586,NaN,1.090407,NaN
2010-01-06,6.315479,6.443004,6.308893,6.417559,552160000,-0.015906,-0.016034,NaN,NaN,NaN,...,NaN,NaN,0.134111,NaN,0.134111,NaN,-0.102080,NaN,2.089751,NaN
2010-01-07,6.303803,6.346312,6.258002,6.338828,477131200,-0.001849,-0.001850,NaN,NaN,NaN,...,NaN,NaN,0.088310,NaN,0.088310,NaN,-0.035025,NaN,1.393159,NaN
2010-01-08,6.345711,6.346310,6.258301,6.295420,447610800,0.006648,0.006626,NaN,NaN,NaN,...,NaN,NaN,0.088010,NaN,0.088010,NaN,0.050291,NaN,1.397997,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-26,254.936371,256.084232,249.336773,251.013651,55969200,0.029713,0.029280,-0.064255,-0.015226,0.179558,...,1.323155,0.947665,6.747459,4.546314,8.504215,4.738337,3.922720,1.924998,2.688084,1.751588
2026-01-27,257.791046,261.464245,257.731160,258.689402,49648300,0.011198,0.011135,-0.058429,-0.016495,0.189899,...,1.123195,0.978318,3.733085,4.570556,6.527874,4.895665,-0.898355,1.897907,1.443076,1.763830
2026-01-28,255.964447,258.379942,254.038018,257.172195,41288000,-0.007086,-0.007111,-0.064041,-0.046141,0.195861,...,0.905259,1.007465,4.341924,4.658012,4.341924,4.983121,-1.207748,1.919294,1.688333,1.800630


In [8]:
# technical indicators
# RSI
df["Price Difference"] = df["Close"].diff()
df["Gains"] = df["Price Difference"].where(df["Price Difference"] > 0, 0)
df["Losses"] = df["Price Difference"].where(df["Price Difference"] < 0, 0).abs()
df["Average Gains"] = df["Gains"].ewm(alpha=1/14, adjust=False).mean()
df["Average Losses"] = df["Losses"].ewm(alpha=1/14, adjust=False).mean()
df["RS"] = df["Average Gains"] / df["Average Losses"]
df["RSI"] = 100 - (100 / (1 + df["RS"]))

# ATR 14
df["14-day Average True Range"] = df["True Range"].ewm(alpha=1/14, adjust=False).mean()

# NATR
df["Normalized Average True Range"] = (df["14-day Average True Range"]/df["Close"]) * 100


# Efficiency Ratio 10-days
df["10-day Efficiency Ratio"] = df["Close"].diff(10).abs() / df["Close"].diff().abs().rolling(10).sum()
# Efficiency Ratio 21-days
df["21-day Efficiency Ratio"] = df["Close"].diff(21).abs() / df["Close"].diff().abs().rolling(21).sum()

# Conviction Ratio
df["Body Up"] = df["Daily Candle Body"].clip(lower=0)
df["Body Down"] = -df["Daily Candle Body"].clip(upper=0)
epsilon = 1e-8
rolling_up = df["Body Up"].rolling(21).mean()
rolling_down = df["Body Down"].rolling(21).mean()
df["Monthly Conviction Ratio"] = rolling_up / (rolling_down + epsilon)



df





,Close,High,Low,Open,Volume,Returns,Log Returns,Monthly Log Returns,Quarterly Log Returns,Semi-annual Log Returns,...,Average Losses,RS,RSI,14-day Average True Range,Normalized Average True Range,10-day Efficiency Ratio,21-day Efficiency Ratio,Body Up,Body Down,Monthly Conviction Ratio
Date,,,,,,,,,,,,,,,,,,,,,
2010-01-04,6.406479,6.421148,6.357685,6.389117,493729600,NaN,NaN,NaN,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.017363,-0.000000,NaN
2010-01-05,6.417557,6.453779,6.383729,6.424143,601904800,0.001729,0.001728,NaN,NaN,NaN,...,0.000000,inf,100.000000,0.070049,1.091526,NaN,NaN,0.000000,0.006586,NaN
2010-01-06,6.315479,6.443004,6.308893,6.417559,552160000,-0.015906,-0.016034,NaN,NaN,NaN,...,0.007291,0.100768,9.154311,0.074625,1.181623,NaN,NaN,0.000000,0.102080,NaN
2010-01-07,6.303803,6.346312,6.258002,6.338828,477131200,-0.001849,-0.001850,NaN,NaN,NaN,...,0.007604,0.089717,8.233040,0.075603,1.199318,NaN,NaN,0.000000,0.035025,NaN
2010-01-08,6.345711,6.346310,6.258301,6.295420,447610800,0.006648,0.006626,NaN,NaN,NaN,...,0.007061,0.513638,33.933985,0.076489,1.205363,NaN,NaN,0.050291,-0.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-26,254.936371,256.084232,249.336773,251.013651,55969200,0.029713,0.029280,-0.064255,-0.015226,0.179558,...,1.350801,0.675864,40.329283,5.122527,2.009335,0.156151,0.397607,3.922720,-0.000000,0.508941
2026-01-27,257.791046,261.464245,257.731160,258.689402,49648300,0.011198,0.011135,-0.058429,-0.016495,0.189899,...,1.254315,0.838427,45.605674,5.222909,2.026024,0.072421,0.352861,0.000000,0.898355,0.439436
2026-01-28,255.964447,258.379942,254.038018,257.172195,41288000,-0.007086,-0.007111,-0.064041,-0.046141,0.195861,...,1.295192,0.753968,42.986417,5.159981,2.015898,0.162495,0.373075,0.000000,1.207748,0.432422


In [9]:
df.columns

Index(['Close', 'High', 'Low', 'Open', 'Volume', 'Returns', 'Log Returns',
       'Monthly Log Returns', 'Quarterly Log Returns',
       'Semi-annual Log Returns', 'Daily Volatility', 'Monthly Volatility',
       'Quarterly Volatility', 'Semi-annual Volatility', 'Volatility Ratio',
       'MA20', 'MA60', 'MA20/MA60', 'Distance from MA20', 'Distance from MA60',
       'VMA20', 'VMA60', 'Relative Volume', 'Volume Flow Ratio', 'Daily Range',
       'Monthly Average Intraday Range', 'True Range',
       'Monthly Average True Range', 'Daily Candle Body',
       'Monthly Average Candle Body', 'Percentage Intraday Range',
       'Monthly Average Percentage Intraday Range', 'Price Difference',
       'Gains', 'Losses', 'Average Gains', 'Average Losses', 'RS', 'RSI',
       '14-day Average True Range', 'Normalized Average True Range',
       '10-day Efficiency Ratio', '21-day Efficiency Ratio', 'Body Up',
       'Body Down', 'Monthly Conviction Ratio'],
      dtype='str')